In [1]:
import os
import glob
import pandas as pd
from typing import Dict, Tuple, Optional

In [2]:
REQUIRED_COLS = [
    "Time_Offset", "CAN_ID", "Data_Length",
    "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Label"
]

def split_each_file_80_20_save(
    in_dir: str,
    save_root: str,
    vehicle_name: str = "Kia",
    train_ratio: float = 0.8,
    sort_by: str = "Time_Offset",
    pattern: str = "*.csv",
    enforce_columns: bool = True,
) -> Dict[str, Dict[str, int]]:
    """
    For each CSV in in_dir:
      - sort by sort_by (if column exists)
      - take first 80% rows as train, last 20% as test
      - save into save_root/{vehicle}_train_80 and save_root/{vehicle}_test_20
      - write split_summary.csv into save_root

    Returns:
      meta dict: per-file row counts
    """
    in_dir = os.path.abspath(in_dir)
    save_root = os.path.abspath(save_root)

    out_train_dir = os.path.join(save_root, f"{vehicle_name}_train_80")
    out_test_dir  = os.path.join(save_root, f"{vehicle_name}_test_20")
    os.makedirs(out_train_dir, exist_ok=True)
    os.makedirs(out_test_dir, exist_ok=True)

    files = sorted(glob.glob(os.path.join(in_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"No CSV files found in: {in_dir}")

    meta = {}

    for fp in files:
        fname = os.path.basename(fp)
        df = pd.read_csv(fp)

        if enforce_columns:
            missing = [c for c in REQUIRED_COLS if c not in df.columns]
            if missing:
                raise ValueError(f"{fname} missing columns: {missing}")

        if sort_by in df.columns:
            df = df.sort_values(sort_by).reset_index(drop=True)

        n = len(df)
        if n < 2:
            train_df = df.copy()
            test_df = df.iloc[0:0].copy()
        else:
            n_train = int(n * train_ratio)
            n_train = max(1, min(n_train, n - 1))  # ensure both parts non-empty
            train_df = df.iloc[:n_train].reset_index(drop=True)
            test_df  = df.iloc[n_train:].reset_index(drop=True)

        train_df.to_csv(os.path.join(out_train_dir, fname), index=False)
        test_df.to_csv(os.path.join(out_test_dir, fname), index=False)

        meta[fname] = {"total": n, "train": len(train_df), "test": len(test_df)}

    # save summary
    summary_path = os.path.join(save_root, f"{vehicle_name}_split_summary.csv")
    pd.DataFrame.from_dict(meta, orient="index").reset_index().rename(columns={"index": "file"}) \
        .to_csv(summary_path, index=False)

    print("Saved:")
    print("  Train:", out_train_dir)
    print("  Test :", out_test_dir)
    print("  Summary:", summary_path)

    return meta

In [3]:
# ---- Run with your paths ----
if __name__ == "__main__":
    meta = split_each_file_80_20_save(
        in_dir="/home/lisa/Arupreza/UIDS/UIDS-II/Split_data/Train/Kia",
        save_root="/home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive",
        vehicle_name="Kia",
        train_ratio=0.8,
        sort_by="Time_Offset",
        enforce_columns=True
    )

Saved:
  Train: /home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/Kia_train_80
  Test : /home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/Kia_test_20
  Summary: /home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/Kia_split_summary.csv


In [11]:
import os, glob, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from scipy.stats import chi2

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

TRAIN_DIR = "/home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/Kia_train_80"
TEST_DIR  = "/home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/Kia_test_20"
CKPT_PATH = "/home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/xiphos_kia_fixed.pt"

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# Data / windowing
NORMAL_STR = "normal"     # label string for normal
WINDOW_SIZE = 3
STRIDE = 3

# GCL (paper-like)
GNN_LAYERS = 2
GNN_OUT = 64
GCL_EPOCHS = 20
GCL_BATCH = 64
GCL_LR = 1e-3
EDGE_PERTURB = 0.1

EMBED_MODE = "avg"  # "phi1" | "phi2" | "avg"

# AHC
GROUP_SIZE = 16
ALPHA = 0.01
AHC_EPOCHS = 20
AHC_LR = 1e-3
AHC_BATCH = 256
AE_H1, AE_H2 = 32, 16
OUT_H1, OUT_H2 = 32, 16

EMBED_BATCH = 256

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE: cuda


In [12]:
REQUIRED_COLS = [
    "Time_Offset","CAN_ID","Data_Length",
    "One","Two","Three","Four","Five","Six","Seven","Eight","Label"
]
PAYLOAD_COLS = ["One","Two","Three","Four","Five","Six","Seven","Eight"]

def parse_hex_or_minus1(x) -> int:
    """
    Parse hex strings like '0340', '0F', 'C5' as base-16.
    Keep -1 if value is '-1' (used as padding for missing bytes).
    """
    if pd.isna(x):
        return -1
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return -1
    if s == "-1":
        return -1
    s2 = s.lower()
    if s2.startswith("0x"):
        return int(s2, 16)
    # IMPORTANT: treat digit-only strings as HEX too (CAN_ID like '0340')
    return int(s2, 16)

def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    out = df.copy()

    out["Time_Offset"] = pd.to_numeric(out["Time_Offset"], errors="coerce").fillna(0.0).astype(np.float64)
    out["Data_Length"] = pd.to_numeric(out["Data_Length"], errors="coerce").fillna(0).astype(np.int64)

    # CAN_ID + bytes are HEX
    out["CAN_ID"] = out["CAN_ID"].apply(parse_hex_or_minus1).astype(np.int64)
    for c in PAYLOAD_COLS:
        out[c] = out[c].apply(parse_hex_or_minus1).astype(np.int64)

    # Label is string => create binary label: 0 normal, 1 attack
    out["Label_str"] = out["Label"].astype(str).str.strip()
    out["Label_bin"] = (out["Label_str"].str.lower() != NORMAL_STR).astype(np.int64)

    # Replace -1 padding bytes with 0 for modeling (DLC keeps the info)
    for c in PAYLOAD_COLS:
        out[c] = out[c].where(out[c] >= 0, 0).astype(np.int64)

    # sort by time
    out = out.sort_values("Time_Offset").reset_index(drop=True)
    return out

In [13]:
from typing import List, Tuple, Dict

def compute_tcg_stats(can_ids: np.ndarray) -> Tuple[int,int,int]:
    unique_ids = np.unique(can_ids)
    num_nodes = int(len(unique_ids))

    edges = [(int(can_ids[i]), int(can_ids[i-1])) for i in range(1, len(can_ids))]  # later -> prev
    edge_set = set(edges)
    num_edges = int(len(edge_set))

    deg: Dict[int,int] = {int(cid): 0 for cid in unique_ids}
    for (u,v) in edge_set:
        if u in deg: deg[u] += 1
        if v in deg: deg[v] += 1
    max_degree = int(max(deg.values())) if deg else 0
    return num_nodes, num_edges, max_degree

def build_crg_adj(can_ids: np.ndarray) -> np.ndarray:
    W = len(can_ids)
    A = np.zeros((W,W), dtype=np.float32)

    # consecutive edges
    for i in range(W-1):
        A[i,i+1] = 1.0
        A[i+1,i] = 1.0

    # same-ID edges
    id_to_idx: Dict[int, List[int]] = {}
    for i,cid in enumerate(can_ids):
        id_to_idx.setdefault(int(cid), []).append(i)

    for idxs in id_to_idx.values():
        if len(idxs) <= 1:
            continue
        for i in range(len(idxs)):
            for j in range(i+1, len(idxs)):
                a,b = idxs[i], idxs[j]
                A[a,b] = 1.0
                A[b,a] = 1.0

    np.fill_diagonal(A, 0.0)
    return A

def make_graph_features(window_df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    can_ids = window_df["CAN_ID"].to_numpy()
    num_nodes, num_edges, max_degree = compute_tcg_stats(can_ids)

    # 13-D node feature:
    # [CAN_ID, 8 bytes, num_nodes, num_edges, max_degree, DLC]
    X = []
    for _, r in window_df.iterrows():
        X.append([
            int(r["CAN_ID"]),
            int(r["One"]), int(r["Two"]), int(r["Three"]), int(r["Four"]),
            int(r["Five"]), int(r["Six"]), int(r["Seven"]), int(r["Eight"]),
            int(num_nodes), int(num_edges), int(max_degree),
            int(r["Data_Length"]),
        ])
    X = np.asarray(X, dtype=np.float32)
    A = build_crg_adj(can_ids)
    return X, A

def build_windows_from_df(df: pd.DataFrame, window_size: int, stride: int):
    X_list, A_list, y_list = [], [], []
    n = len(df)
    for start in range(0, n - window_size + 1, stride):
        wdf = df.iloc[start:start+window_size]
        X, A = make_graph_features(wdf)
        # window label: attack if ANY row is attack (Label_bin==1)
        y = int(np.any(wdf["Label_bin"].to_numpy() == 1))
        X_list.append(X); A_list.append(A); y_list.append(y)
    return X_list, A_list, np.asarray(y_list, dtype=np.int64)

def build_windows_from_folder(folder: str, window_size: int, stride: int):
    paths = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if not paths:
        raise FileNotFoundError(f"No CSVs found in {folder}")

    X_all, A_all, y_all = [], [], []
    for p in paths:
        df = preprocess_df(pd.read_csv(p))
        X_list, A_list, y = build_windows_from_df(df, window_size, stride)
        X_all.extend(X_list); A_all.extend(A_list); y_all.append(y)

    y_all = np.concatenate(y_all, axis=0) if len(y_all) else np.array([], dtype=np.int64)
    return X_all, A_all, y_all

In [14]:
def fit_minmax_scaler_X_list(X_list):
    X = np.concatenate([x for x in X_list], axis=0)  # (total_nodes, 13)
    mn = X.min(axis=0)
    mx = X.max(axis=0)
    return mn, mx

def apply_minmax_scaler_X_list(X_list, mn, mx):
    denom = np.maximum(mx - mn, 1e-12)
    out = []
    for x in X_list:
        out.append(((x - mn) / denom).astype(np.float32))
    return out

In [15]:
class GraphWindowDataset(Dataset):
    def __init__(self, X_list, A_list, y):
        self.X_list = X_list
        self.A_list = A_list
        self.y = y
    def __len__(self): return len(self.X_list)
    def __getitem__(self, idx):
        X = torch.from_numpy(self.X_list[idx]).float()
        A = torch.from_numpy(self.A_list[idx]).float()
        y = torch.tensor(int(self.y[idx]), dtype=torch.long)
        return X, A, y

class BatchedGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)
    def forward(self, x, adj):
        B,N,_ = x.shape
        I = torch.eye(N, device=x.device).unsqueeze(0).expand(B,N,N)
        A = adj + I
        deg = A.sum(dim=-1).clamp(min=1e-8)
        deg_inv_sqrt = deg.pow(-0.5)
        A_norm = deg_inv_sqrt.unsqueeze(-1) * A * deg_inv_sqrt.unsqueeze(-2)
        h = torch.matmul(A_norm, x)
        return self.lin(h)

class GNNConcatReadoutEncoder(nn.Module):
    # z = CONCAT(h^1..h^K), Z = CONCAT(mean(h^1)..mean(h^K))
    def __init__(self, in_dim, out_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        dims = [in_dim] + [out_dim]*num_layers
        self.layers = nn.ModuleList([BatchedGCNLayer(dims[i], dims[i+1]) for i in range(num_layers)])
    def forward(self, x, adj):
        hs = []
        h = x
        for k, layer in enumerate(self.layers):
            h = layer(h, adj)
            if k < self.num_layers - 1:
                h = F.relu(h)
            hs.append(h)
        z = torch.cat(hs, dim=-1)  # (B,N,K*out_dim)
        Z = torch.cat([hk.mean(dim=1) for hk in hs], dim=-1)  # (B,K*out_dim)
        return z, Z

def edge_perturb_toggle(adj: torch.Tensor, p: float) -> torch.Tensor:
    if p <= 0:
        return adj
    B,N,_ = adj.shape
    out = adj.clone()
    upper_mask = torch.triu(torch.ones((N,N), device=adj.device, dtype=torch.bool), diagonal=1)
    flip = (torch.rand((B,N,N), device=adj.device) < p) & upper_mask
    out[flip] = 1.0 - out[flip]
    upper = torch.triu(out, diagonal=1)
    out = upper + upper.transpose(1,2)
    eye = torch.eye(N, device=adj.device).unsqueeze(0).expand(B,N,N)
    out = out * (1.0 - eye)
    return out

class TwoLayerMLP_PReLU(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, out_dim)
        self.act = nn.PReLU()
        self.fc2 = nn.Linear(out_dim, out_dim)
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class BilinearDiscriminator(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.W = nn.Parameter(torch.empty(dim, dim))
        nn.init.xavier_uniform_(self.W)
    def forward(self, a, b):
        if a.dim() == 3:
            aW = torch.matmul(a, self.W)
            return torch.sum(aW * b, dim=-1)
        aW = torch.matmul(a, self.W)
        return torch.sum(aW * b, dim=-1)

def jsd_mi_loss(pos_scores, neg_scores):
    return F.softplus(-pos_scores).mean() + F.softplus(neg_scores).mean()

def train_gcl(train_loader, in_dim, out_dim, num_layers, epochs, lr, edge_p, device):
    # two encoders
    enc1 = GNNConcatReadoutEncoder(in_dim, out_dim, num_layers).to(device)
    enc2 = GNNConcatReadoutEncoder(in_dim, out_dim, num_layers).to(device)

    D = num_layers * out_dim
    proj_l = TwoLayerMLP_PReLU(D, D).to(device)
    proj_g = TwoLayerMLP_PReLU(D, D).to(device)
    disc_l = BilinearDiscriminator(D).to(device)
    disc_g = BilinearDiscriminator(D).to(device)

    opt = torch.optim.Adam(
        list(enc1.parameters()) + list(enc2.parameters()) +
        list(proj_l.parameters()) + list(proj_g.parameters()) +
        list(disc_l.parameters()) + list(disc_g.parameters()),
        lr=lr
    )

    for ep in range(epochs):
        losses = []
        for X,A,_y in train_loader:
            X = X.to(device); A = A.to(device)
            A1 = edge_perturb_toggle(A, edge_p)
            A2 = edge_perturb_toggle(A, edge_p)

            z1,Z1 = enc1(X,A1)
            z2,Z2 = enc2(X,A2)

            z1p = proj_l(z1); z2p = proj_l(z2)
            Z1p = proj_g(Z1); Z2p = proj_g(Z2)

            shuf = torch.randperm(z2p.shape[0], device=device)
            z2n = z2p[shuf]; Z2n = Z2p[shuf]

            s_pos_l = disc_l(z1p, z2p)
            s_neg_l = disc_l(z1p, z2n)
            s_pos_g = disc_g(Z1p, Z2p)
            s_neg_g = disc_g(Z1p, Z2n)

            loss = jsd_mi_loss(s_pos_l, s_neg_l) + jsd_mi_loss(s_pos_g, s_neg_g)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())

        print(f"[GCL] epoch {ep+1:03d}/{epochs} loss={float(np.mean(losses)):.4f}")

    return enc1, enc2, proj_l, proj_g, disc_l, disc_g

@torch.no_grad()
def embed_graphs(enc1, enc2, loader, device, mode="avg"):
    enc1.eval(); enc2.eval()
    outs = []
    for X,A,_y in loader:
        X = X.to(device); A = A.to(device)
        _z1, Z1 = enc1(X,A)
        _z2, Z2 = enc2(X,A)
        if mode == "phi1":
            Z = Z1
        elif mode == "phi2":
            Z = Z2
        else:
            Z = 0.5*(Z1 + Z2)
        outs.append(Z.cpu().numpy())
    return np.concatenate(outs, axis=0).astype(np.float64)

In [16]:
def corr_distance_matrix(U: np.ndarray) -> np.ndarray:
    C = np.corrcoef(U, rowvar=False)
    C = np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)
    D = 1.0 - C
    np.fill_diagonal(D, 0.0)
    return D.astype(np.float64)

def split_cluster_tree_to_max_size(Z: np.ndarray, d: int, max_size: int):
    children = {d+k: (int(Z[k,0]), int(Z[k,1])) for k in range(Z.shape[0])}
    root = d + Z.shape[0] - 1

    def leaves(node):
        if node < d: return [node]
        a,b = children[node]
        return leaves(a) + leaves(b)

    stack = [root]
    clusters = []
    while stack:
        node = stack.pop()
        lf = leaves(node)
        if len(lf) <= max_size:
            clusters.append(sorted(lf))
        else:
            a,b = children[node]
            stack.append(a); stack.append(b)

    clusters.sort(key=lambda x: x[0])
    return clusters

class ThreeLayerAE(nn.Module):
    def __init__(self, in_dim, h1, h2):
        super().__init__()
        self.enc1 = nn.Linear(in_dim, h1)
        self.enc2 = nn.Linear(h1, h2)
        self.dec  = nn.Linear(h2, in_dim)
    def forward(self, x):
        z = F.relu(self.enc1(x))
        z = F.relu(self.enc2(z))
        return self.dec(z)

class AHCModel:
    def __init__(self, groups, group_size, ensemble, output_ae,
                 rmse_min, rmse_max, mu, cov_inv, thresh):
        self.groups = groups
        self.group_size = group_size
        self.ensemble = ensemble
        self.output_ae = output_ae
        self.rmse_min = rmse_min
        self.rmse_max = rmse_max
        self.mu = mu
        self.cov_inv = cov_inv
        self.thresh = thresh

def fit_ahc(U_train_normal: np.ndarray, group_size: int, alpha: float,
            ae_h1:int, ae_h2:int, out_h1:int, out_h2:int,
            epochs:int, lr:float, batch_size:int, device:str):

    n, d = U_train_normal.shape
    Dm = corr_distance_matrix(U_train_normal)
    Z = linkage(squareform(Dm, checks=False), method="average")
    groups = split_cluster_tree_to_max_size(Z, d=d, max_size=group_size)
    m = len(groups)

    def map_groups(U):
        V = np.zeros((U.shape[0], m, group_size), dtype=np.float32)
        for j, idxs in enumerate(groups):
            vals = U[:, idxs].astype(np.float32)
            V[:, j, :vals.shape[1]] = vals
        return V

    V_train = map_groups(U_train_normal)
    Vt = torch.from_numpy(V_train).to(device)

    ensemble = [ThreeLayerAE(group_size, ae_h1, ae_h2).to(device) for _ in range(m)]
    opts = [torch.optim.Adam(ae.parameters(), lr=lr) for ae in ensemble]
    mse = nn.MSELoss()

    for _ep in range(epochs):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            vb = Vt[idx]
            for j in range(m):
                x = vb[:, j, :]
                recon = ensemble[j](x)
                loss = mse(recon, x)
                opts[j].zero_grad()
                loss.backward()
                opts[j].step()

    with torch.no_grad():
        rmse_ens = []
        for j in range(m):
            x = Vt[:, j, :]
            recon = ensemble[j](x)
            e = torch.sqrt(torch.mean((recon-x)**2, dim=1) + 1e-12)
            rmse_ens.append(e.cpu().numpy())
        rmse_ens = np.stack(rmse_ens, axis=1).astype(np.float64)

    rmse_min = rmse_ens.min(axis=0)
    rmse_max = rmse_ens.max(axis=0)
    denom = np.maximum(rmse_max - rmse_min, 1e-12)
    rmse_norm = ((rmse_ens - rmse_min) / denom).astype(np.float32)

    out_ae = ThreeLayerAE(m, out_h1, out_h2).to(device)
    out_opt = torch.optim.Adam(out_ae.parameters(), lr=lr)
    R = torch.from_numpy(rmse_norm).to(device)

    for _ep in range(epochs):
        perm = torch.randperm(n, device=device)
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            rb = R[idx]
            recon = out_ae(rb)
            loss = mse(recon, rb)
            out_opt.zero_grad()
            loss.backward()
            out_opt.step()

    with torch.no_grad():
        recon = out_ae(R)
        rmse_out = torch.sqrt(torch.mean((recon-R)**2, dim=1) + 1e-12).cpu().numpy().astype(np.float64)

    RM = np.concatenate([rmse_ens, rmse_out[:, None]], axis=1)
    mu = RM.mean(axis=0)
    cov = np.cov(RM, rowvar=False) + np.eye(RM.shape[1]) * 1e-6
    cov_inv = np.linalg.inv(cov)
    thresh = float(chi2.ppf(1.0 - alpha, df=RM.shape[1]))
    return AHCModel(groups, group_size, ensemble, out_ae, rmse_min, rmse_max, mu, cov_inv, thresh)

def ahc_scores(ahc: AHCModel, U: np.ndarray, device: str) -> np.ndarray:
    n, d = U.shape
    groups, s = ahc.groups, ahc.group_size
    m = len(groups)

    V = np.zeros((n, m, s), dtype=np.float32)
    for j, idxs in enumerate(groups):
        vals = U[:, idxs].astype(np.float32)
        V[:, j, :vals.shape[1]] = vals
    Vt = torch.from_numpy(V).to(device)

    with torch.no_grad():
        rmse_ens = []
        for j in range(m):
            x = Vt[:, j, :]
            recon = ahc.ensemble[j](x)
            e = torch.sqrt(torch.mean((recon-x)**2, dim=1) + 1e-12)
            rmse_ens.append(e.cpu().numpy())
        rmse_ens = np.stack(rmse_ens, axis=1).astype(np.float64)

    denom = np.maximum(ahc.rmse_max - ahc.rmse_min, 1e-12)
    rmse_norm = ((rmse_ens - ahc.rmse_min) / denom).astype(np.float32)

    R = torch.from_numpy(rmse_norm).to(device)
    with torch.no_grad():
        recon = ahc.output_ae(R)
        rmse_out = torch.sqrt(torch.mean((recon-R)**2, dim=1) + 1e-12).cpu().numpy().astype(np.float64)

    RM = np.concatenate([rmse_ens, rmse_out[:, None]], axis=1)
    diff = RM - ahc.mu[None, :]
    scores = np.einsum("ni,ij,nj->n", diff, ahc.cov_inv, diff)
    return scores

def report(y_true, y_pred, name="TEST"):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    print(f"[{name}] acc={acc:.4f} prec={p:.4f} rec={r:.4f} f1={f1:.4f}")

In [17]:
# 1) Build windows
X_tr, A_tr, y_tr = build_windows_from_folder(TRAIN_DIR, WINDOW_SIZE, STRIDE)
X_te, A_te, y_te = build_windows_from_folder(TEST_DIR,  WINDOW_SIZE, STRIDE)

print("Train windows:", len(y_tr), "attack ratio:", float(y_tr.mean()) if len(y_tr) else None)
print("Test  windows:", len(y_te), "attack ratio:", float(y_te.mean()) if len(y_te) else None)

# 2) Fit scaler on TRAIN only, apply to train+test (recommended)
mn, mx = fit_minmax_scaler_X_list(X_tr)
X_tr = apply_minmax_scaler_X_list(X_tr, mn, mx)
X_te = apply_minmax_scaler_X_list(X_te, mn, mx)

# 3) Loaders
train_ds = GraphWindowDataset(X_tr, A_tr, y_tr)
test_ds  = GraphWindowDataset(X_te, A_te, y_te)

train_loader = DataLoader(train_ds, batch_size=GCL_BATCH, shuffle=True, drop_last=False)
embed_train_loader = DataLoader(train_ds, batch_size=EMBED_BATCH, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=EMBED_BATCH, shuffle=False, drop_last=False)

# 4) Train GCL (contrastive)
enc1, enc2, proj_l, proj_g, disc_l, disc_g = train_gcl(
    train_loader=train_loader,
    in_dim=13,
    out_dim=GNN_OUT,
    num_layers=GNN_LAYERS,
    epochs=GCL_EPOCHS,
    lr=GCL_LR,
    edge_p=EDGE_PERTURB,
    device=DEVICE
)

# 5) Embed train/test
U_tr = embed_graphs(enc1, enc2, embed_train_loader, DEVICE, mode=EMBED_MODE)
U_te = embed_graphs(enc1, enc2, test_loader, DEVICE, mode=EMBED_MODE)

# 6) Fit AHC on normal-only train windows
normal_mask = (y_tr == 0)
print("Normal train windows:", int(normal_mask.sum()), "/", len(y_tr))
if normal_mask.sum() < 10:
    raise RuntimeError("Not enough normal windows to fit AHC. Increase data or adjust NORMAL_STR.")

ahc = fit_ahc(
    U_train_normal=U_tr[normal_mask],
    group_size=GROUP_SIZE,
    alpha=ALPHA,
    ae_h1=AE_H1, ae_h2=AE_H2,
    out_h1=OUT_H1, out_h2=OUT_H2,
    epochs=AHC_EPOCHS,
    lr=AHC_LR,
    batch_size=AHC_BATCH,
    device=DEVICE
)

print("SGM threshold:", ahc.thresh)

# 7) Test
scores_te = ahc_scores(ahc, U_te, DEVICE)
yhat_te = (scores_te > ahc.thresh).astype(np.int64)
report(y_te, yhat_te, name="TEST")

# 8) Save checkpoint
ckpt = {
    "config": {
        "NORMAL_STR": NORMAL_STR,
        "WINDOW_SIZE": WINDOW_SIZE,
        "STRIDE": STRIDE,
        "GNN_LAYERS": GNN_LAYERS,
        "GNN_OUT": GNN_OUT,
        "EDGE_PERTURB": EDGE_PERTURB,
        "EMBED_MODE": EMBED_MODE,
        "GROUP_SIZE": GROUP_SIZE,
        "ALPHA": ALPHA,
        "scaler_min": mn,
        "scaler_max": mx,
    },
    "enc1": enc1.state_dict(),
    "enc2": enc2.state_dict(),
    "proj_l": proj_l.state_dict(),
    "proj_g": proj_g.state_dict(),
    "disc_l": disc_l.state_dict(),
    "disc_g": disc_g.state_dict(),
    "ahc": {
        "groups": ahc.groups,
        "group_size": ahc.group_size,
        "rmse_min": ahc.rmse_min,
        "rmse_max": ahc.rmse_max,
        "mu": ahc.mu,
        "cov_inv": ahc.cov_inv,
        "thresh": ahc.thresh,
        "ensemble_state": [m.state_dict() for m in ahc.ensemble],
        "output_state": ahc.output_ae.state_dict(),
        "AE_H1": AE_H1, "AE_H2": AE_H2,
        "OUT_H1": OUT_H1, "OUT_H2": OUT_H2,
    }
}
torch.save(ckpt, CKPT_PATH)
print("Saved:", CKPT_PATH)

Train windows: 2068556 attack ratio: 0.3127814765469245
Test  windows: 517133 attack ratio: 0.31274159645584404
[GCL] epoch 001/20 loss=0.4524
[GCL] epoch 002/20 loss=0.3467
[GCL] epoch 003/20 loss=0.3231
[GCL] epoch 004/20 loss=0.3110
[GCL] epoch 005/20 loss=0.3029
[GCL] epoch 006/20 loss=0.2974
[GCL] epoch 007/20 loss=0.2930
[GCL] epoch 008/20 loss=0.2908
[GCL] epoch 009/20 loss=0.2869
[GCL] epoch 010/20 loss=0.2848
[GCL] epoch 011/20 loss=0.2824
[GCL] epoch 012/20 loss=0.2798
[GCL] epoch 013/20 loss=0.2803
[GCL] epoch 014/20 loss=0.2780
[GCL] epoch 015/20 loss=0.2782
[GCL] epoch 016/20 loss=0.2771
[GCL] epoch 017/20 loss=0.2758
[GCL] epoch 018/20 loss=0.2735
[GCL] epoch 019/20 loss=0.2736
[GCL] epoch 020/20 loss=0.2730
Normal train windows: 1421550 / 2068556


/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/lisa/Arupreza/UIDS-II/uids/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


SGM threshold: 31.999926908815176
[TEST] acc=0.7039 prec=0.8793 rec=0.0615 f1=0.1150
Saved: /home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/xiphos_kia_fixed.pt


In [25]:
import os, glob
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ============================================================
# 0) CONFIG
# ============================================================
CKPT_PATH = "/home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/xiphos_kia_fixed.pt"
TEST_DIR  = "/home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/Kia_test_20"
SAVE_CSV  = "/home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/kia_test_predictions.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ============================================================
# 1) DATA PARSING (YOUR HEX FORMAT)
# ============================================================
REQUIRED_COLS = [
    "Time_Offset","CAN_ID","Data_Length",
    "One","Two","Three","Four","Five","Six","Seven","Eight","Label"
]
PAYLOAD_COLS = ["One","Two","Three","Four","Five","Six","Seven","Eight"]

def parse_hex_or_minus1(x) -> int:
    """
    Parse hex strings like '0340', '0F', 'C5' as base-16.
    Keep -1 if value is '-1' (padding).
    IMPORTANT: treat digit-only strings as HEX too.
    """
    if pd.isna(x):
        return -1
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return -1
    if s == "-1":
        return -1
    s2 = s.lower()
    if s2.startswith("0x"):
        return int(s2, 16)
    return int(s2, 16)

def preprocess_df(df: pd.DataFrame, normal_str: str = "normal") -> pd.DataFrame:
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    out = df.copy()
    out["Time_Offset"] = pd.to_numeric(out["Time_Offset"], errors="coerce").fillna(0.0).astype(np.float64)
    out["Data_Length"] = pd.to_numeric(out["Data_Length"], errors="coerce").fillna(0).astype(np.int64)

    out["CAN_ID"] = out["CAN_ID"].apply(parse_hex_or_minus1).astype(np.int64)
    for c in PAYLOAD_COLS:
        out[c] = out[c].apply(parse_hex_or_minus1).astype(np.int64)

    out["Label_str"] = out["Label"].astype(str).str.strip()
    out["Label_bin"] = (out["Label_str"].str.lower() != normal_str).astype(np.int64)

    # payload padding -1 -> 0 (DLC carries real length)
    for c in PAYLOAD_COLS:
        out[c] = out[c].where(out[c] >= 0, 0).astype(np.int64)

    out = out.sort_values("Time_Offset").reset_index(drop=True)
    return out

# ============================================================
# 2) TCG/CRG + WINDOW GRAPHS (per-file, no leakage)
# ============================================================
from typing import List, Tuple, Dict

def compute_tcg_stats(can_ids: np.ndarray) -> Tuple[int,int,int]:
    unique_ids = np.unique(can_ids)
    num_nodes = int(len(unique_ids))

    edges = [(int(can_ids[i]), int(can_ids[i-1])) for i in range(1, len(can_ids))]
    edge_set = set(edges)
    num_edges = int(len(edge_set))

    deg: Dict[int,int] = {int(cid): 0 for cid in unique_ids}
    for (u,v) in edge_set:
        if u in deg: deg[u] += 1
        if v in deg: deg[v] += 1
    max_degree = int(max(deg.values())) if deg else 0
    return num_nodes, num_edges, max_degree

def build_crg_adj(can_ids: np.ndarray) -> np.ndarray:
    W = len(can_ids)
    A = np.zeros((W,W), dtype=np.float32)

    # consecutive edges
    for i in range(W-1):
        A[i,i+1] = 1.0
        A[i+1,i] = 1.0

    # same-ID edges
    id_to_idx: Dict[int, List[int]] = {}
    for i,cid in enumerate(can_ids):
        id_to_idx.setdefault(int(cid), []).append(i)

    for idxs in id_to_idx.values():
        if len(idxs) <= 1:
            continue
        for i in range(len(idxs)):
            for j in range(i+1, len(idxs)):
                a,b = idxs[i], idxs[j]
                A[a,b] = 1.0
                A[b,a] = 1.0

    np.fill_diagonal(A, 0.0)
    return A

def make_graph_features(window_df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    can_ids = window_df["CAN_ID"].to_numpy()
    num_nodes, num_edges, max_degree = compute_tcg_stats(can_ids)

    # 13-D node feature:
    # [CAN_ID, 8 bytes, num_nodes, num_edges, max_degree, DLC]
    X = []
    for _, r in window_df.iterrows():
        X.append([
            int(r["CAN_ID"]),
            int(r["One"]), int(r["Two"]), int(r["Three"]), int(r["Four"]),
            int(r["Five"]), int(r["Six"]), int(r["Seven"]), int(r["Eight"]),
            int(num_nodes), int(num_edges), int(max_degree),
            int(r["Data_Length"]),
        ])
    X = np.asarray(X, dtype=np.float32)
    A = build_crg_adj(can_ids)
    return X, A

def build_windows_from_df(df: pd.DataFrame, window_size: int, stride: int):
    X_list, A_list, y_list = [], [], []
    n = len(df)
    for start in range(0, n - window_size + 1, stride):
        wdf = df.iloc[start:start+window_size]
        X, A = make_graph_features(wdf)
        y = int(np.any(wdf["Label_bin"].to_numpy() == 1))  # window-level
        X_list.append(X); A_list.append(A); y_list.append(y)
    return X_list, A_list, np.asarray(y_list, dtype=np.int64)

def build_windows_from_folder(folder: str, window_size: int, stride: int, normal_str: str):
    paths = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if not paths:
        raise FileNotFoundError(f"No CSVs found in {folder}")

    X_all, A_all, y_all = [], [], []
    src_all = []
    for p in paths:
        df = preprocess_df(pd.read_csv(p), normal_str=normal_str)
        X_list, A_list, y = build_windows_from_df(df, window_size, stride)
        X_all.extend(X_list); A_all.extend(A_list); y_all.append(y)
        src_all.extend([os.path.basename(p)] * len(y))

    y_all = np.concatenate(y_all, axis=0) if len(y_all) else np.array([], dtype=np.int64)
    return X_all, A_all, y_all, np.asarray(src_all)

# ============================================================
# 3) SCALER
# ============================================================
def apply_minmax_scaler_X_list(X_list, mn, mx):
    mn = np.asarray(mn, dtype=np.float32)
    mx = np.asarray(mx, dtype=np.float32)
    denom = np.maximum(mx - mn, 1e-12)
    out = []
    for x in X_list:
        out.append(((x - mn) / denom).astype(np.float32))
    return out

# ============================================================
# 4) MODEL DEFINITIONS (encoders + AE)
# ============================================================
class BatchedGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim)
    def forward(self, x, adj):
        B,N,_ = x.shape
        I = torch.eye(N, device=x.device).unsqueeze(0).expand(B,N,N)
        A = adj + I
        deg = A.sum(dim=-1).clamp(min=1e-8)
        deg_inv_sqrt = deg.pow(-0.5)
        A_norm = deg_inv_sqrt.unsqueeze(-1) * A * deg_inv_sqrt.unsqueeze(-2)
        h = torch.matmul(A_norm, x)
        return self.lin(h)

class GNNConcatReadoutEncoder(nn.Module):
    def __init__(self, in_dim, out_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        dims = [in_dim] + [out_dim]*num_layers
        self.layers = nn.ModuleList([BatchedGCNLayer(dims[i], dims[i+1]) for i in range(num_layers)])
    def forward(self, x, adj):
        hs = []
        h = x
        for k, layer in enumerate(self.layers):
            h = layer(h, adj)
            if k < self.num_layers - 1:
                h = F.relu(h)
            hs.append(h)
        z = torch.cat(hs, dim=-1)  # (B,N,K*out_dim)
        Z = torch.cat([hk.mean(dim=1) for hk in hs], dim=-1)  # (B,K*out_dim)
        return z, Z

class ThreeLayerAE(nn.Module):
    def __init__(self, in_dim, h1, h2):
        super().__init__()
        self.enc1 = nn.Linear(in_dim, h1)
        self.enc2 = nn.Linear(h1, h2)
        self.dec  = nn.Linear(h2, in_dim)
    def forward(self, x):
        z = F.relu(self.enc1(x))
        z = F.relu(self.enc2(z))
        return self.dec(z)

# ============================================================
# 5) INFERENCE: embed + AHC scoring
# ============================================================
@torch.no_grad()
def embed_graphs(enc1, enc2, X_list, A_list, batch_size, device, mode="avg"):
    enc1.eval(); enc2.eval()
    outs = []
    n = len(X_list)
    for i in range(0, n, batch_size):
        Xb = torch.from_numpy(np.stack(X_list[i:i+batch_size], axis=0)).to(device)  # (B,W,13)
        Ab = torch.from_numpy(np.stack(A_list[i:i+batch_size], axis=0)).to(device)  # (B,W,W)
        _z1, Z1 = enc1(Xb, Ab)
        _z2, Z2 = enc2(Xb, Ab)
        if mode == "phi1":
            Z = Z1
        elif mode == "phi2":
            Z = Z2
        else:
            Z = 0.5*(Z1 + Z2)
        outs.append(Z.cpu().numpy())
    return np.concatenate(outs, axis=0).astype(np.float64)

@torch.no_grad()
def ahc_scores(U, groups, group_size, ensemble_models, output_ae,
               rmse_min, rmse_max, mu, cov_inv, device):
    n, d = U.shape
    m = len(groups)

    V = np.zeros((n, m, group_size), dtype=np.float32)
    for j, idxs in enumerate(groups):
        vals = U[:, idxs].astype(np.float32)
        V[:, j, :vals.shape[1]] = vals
    Vt = torch.from_numpy(V).to(device)

    rmse_ens = []
    for j in range(m):
        x = Vt[:, j, :]
        recon = ensemble_models[j](x)
        e = torch.sqrt(torch.mean((recon - x)**2, dim=1) + 1e-12)
        rmse_ens.append(e.cpu().numpy())
    rmse_ens = np.stack(rmse_ens, axis=1).astype(np.float64)

    rmse_min = np.asarray(rmse_min, dtype=np.float64)
    rmse_max = np.asarray(rmse_max, dtype=np.float64)
    denom = np.maximum(rmse_max - rmse_min, 1e-12)
    rmse_norm = ((rmse_ens - rmse_min) / denom).astype(np.float32)

    R = torch.from_numpy(rmse_norm).to(device)
    recon = output_ae(R)
    rmse_out = torch.sqrt(torch.mean((recon - R)**2, dim=1) + 1e-12).cpu().numpy().astype(np.float64)

    RM = np.concatenate([rmse_ens, rmse_out[:, None]], axis=1)  # (n,m+1)

    diff = RM - np.asarray(mu, dtype=np.float64)[None, :]
    cov_inv = np.asarray(cov_inv, dtype=np.float64)
    scores = np.einsum("ni,ij,nj->n", diff, cov_inv, diff)  # Mahalanobis^2
    return scores

def evaluate_binary(y_true, y_pred, name="TEST"):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    print(f"[{name}] acc={acc:.4f} prec={p:.4f} rec={r:.4f} f1={f1:.4f}")

# ============================================================
# 6) MAIN: LOAD CKPT (PyTorch 2.6 fix) + RUN
# ============================================================
def run_inference_on_folder(ckpt_path: str, folder: str, batch_size: int = 256,
                            device: str = None, save_csv: str = None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # PyTorch 2.6 fix: checkpoint contains numpy objects -> weights_only=False
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    cfg = ckpt["config"]
    normal_str = cfg["NORMAL_STR"]
    window_size = cfg["WINDOW_SIZE"]
    stride = cfg["STRIDE"]
    num_layers = cfg["GNN_LAYERS"]
    out_dim = cfg["GNN_OUT"]
    embed_mode = cfg["EMBED_MODE"]

    thresh = ckpt["ahc"]["thresh"]

    # windows
    X_list, A_list, y_true, src = build_windows_from_folder(folder, window_size, stride, normal_str)

    # scaler
    mn = cfg["scaler_min"]
    mx = cfg["scaler_max"]
    X_list = apply_minmax_scaler_X_list(X_list, mn, mx)

    # encoders
    enc1 = GNNConcatReadoutEncoder(13, out_dim, num_layers).to(device)
    enc2 = GNNConcatReadoutEncoder(13, out_dim, num_layers).to(device)
    enc1.load_state_dict(ckpt["enc1"])
    enc2.load_state_dict(ckpt["enc2"])
    enc1.eval(); enc2.eval()

    # AHC AEs
    groups = ckpt["ahc"]["groups"]
    group_size = ckpt["ahc"]["group_size"]
    m = len(groups)

    AE_H1, AE_H2 = ckpt["ahc"]["AE_H1"], ckpt["ahc"]["AE_H2"]
    OUT_H1, OUT_H2 = ckpt["ahc"]["OUT_H1"], ckpt["ahc"]["OUT_H2"]

    ensemble_models = []
    for st in ckpt["ahc"]["ensemble_state"]:
        ae = ThreeLayerAE(group_size, AE_H1, AE_H2).to(device)
        ae.load_state_dict(st)
        ae.eval()
        ensemble_models.append(ae)

    output_ae = ThreeLayerAE(m, OUT_H1, OUT_H2).to(device)
    output_ae.load_state_dict(ckpt["ahc"]["output_state"])
    output_ae.eval()

    # embed + score
    U = embed_graphs(enc1, enc2, X_list, A_list, batch_size=batch_size, device=device, mode=embed_mode)
    scores = ahc_scores(
        U, groups, group_size, ensemble_models, output_ae,
        ckpt["ahc"]["rmse_min"], ckpt["ahc"]["rmse_max"],
        ckpt["ahc"]["mu"], ckpt["ahc"]["cov_inv"], device
    )
    y_pred = (scores > thresh).astype(np.int64)

    if len(y_true) == len(y_pred) and len(y_true) > 0:
        evaluate_binary(y_true, y_pred, name=os.path.basename(folder))

    if save_csv:
        out = pd.DataFrame({
            "source_file": src,
            "window_index": np.arange(len(y_pred)),
            "y_true": y_true,
            "y_pred": y_pred,
            "score": scores,
            "threshold": float(thresh),
        })
        out.to_csv(save_csv, index=False)
        print("Saved:", save_csv)

    return y_true, y_pred, scores, float(thresh)

DEVICE: cuda


In [26]:
# ============================================================
# RUN
# ============================================================
y_true, y_pred, scores, thr = run_inference_on_folder(
    ckpt_path=CKPT_PATH,
    folder=TEST_DIR,
    batch_size=256,
    device=DEVICE,
    save_csv=SAVE_CSV
)

print("Done. Threshold =", thr, " Num windows =", len(y_pred))

[Kia_test_20] acc=0.7039 prec=0.8793 rec=0.0615 f1=0.1150
Saved: /home/lisa/Arupreza/UIDS/UIDS-II/Jiang Adaptive/kia_test_predictions.csv
Done. Threshold = 31.999926908815176  Num windows = 517133
